In [1]:
import heapq

class PuzzleState:
    def __init__(self, state, parent, move, depth, cost):
        self.state = state
        self.parent = parent
        self.move = move
        self.depth = depth
        self.cost = cost
        
    def __lt__(self, other):
        return self.cost < other.cost

class NPuzzleAStar:
    def __init__(self, n):
        self.n = n
        self.goal = tuple(list(range(1, n*n)) + [0]) # Đích: (1,2,3,...,0)
        
    # Heuristic: Khoảng cách Manhattan (Tối ưu hơn cách đếm ô sai vị trí của code mẫu)
    def manhattan_distance(self, state):
        dist = 0
        for i, val in enumerate(state):
            if val != 0:
                target_x, target_y = (val - 1) // self.n, (val - 1) % self.n
                curr_x, curr_y = i // self.n, i % self.n
                dist += abs(target_x - curr_x) + abs(target_y - curr_y)
        return dist

    def get_neighbors(self, state):
        neighbors = []
        zero_idx = state.index(0)
        x, y = zero_idx // self.n, zero_idx % self.n
        
        # Các hướng di chuyển: Lên, Xuống, Trái, Phải
        moves = {'Up': (-1, 0), 'Down': (1, 0), 'Left': (0, -1), 'Right': (0, 1)}
        
        for move_name, (dx, dy) in moves.items():
            nx, ny = x + dx, y + dy
            if 0 <= nx < self.n and 0 <= ny < self.n:
                new_idx = nx * self.n + ny
                new_state = list(state)
                # Đổi chỗ ô trống và ô kề
                new_state[zero_idx], new_state[new_idx] = new_state[new_idx], new_state[zero_idx]
                neighbors.append((tuple(new_state), move_name))
        return neighbors

    def solve(self, initial_state):
        # Biến đổi mảng 2D đầu vào thành tuple 1 chiều
        flat_initial = tuple(val for row in initial_state for val in row)
        
        open_list = []
        closed_set = set()
        
        start_node = PuzzleState(flat_initial, None, "Start", 0, self.manhattan_distance(flat_initial))
        heapq.heappush(open_list, start_node)
        
        while open_list:
            current_node = heapq.heappop(open_list)
            
            if current_node.state == self.goal:
                return self.build_path(current_node)
                
            closed_set.add(current_node.state)
            
            for next_state, move_name in self.get_neighbors(current_node.state):
                if next_state in closed_set:
                    continue
                    
                g = current_node.depth + 1
                h = self.manhattan_distance(next_state)
                f = g + h
                
                child_node = PuzzleState(next_state, current_node, move_name, g, f)
                heapq.heappush(open_list, child_node)
                
        return None # Không tìm thấy đường

    def build_path(self, node):
        path = []
        while node:
            path.append((node.move, node.state))
            node = node.parent
        return path[::-1]

    def print_state(self, state):
        for i in range(self.n):
            row = state[i*self.n : (i+1)*self.n]
            print(" ".join(f"{str(x):>2}" for x in row))
        print("-" * 10)

# Kịch bản chạy thử (Áp dụng cho n=4)
if __name__ == "__main__":
    initial = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 0, 11],
        [13, 14, 15, 12]
    ]
    
    solver = NPuzzleAStar(n=4)
    path = solver.solve(initial)
    
    if path:
        print(f"Đã tìm thấy đường đi sau {len(path)-1} bước:")
        for step, state in path:
            print(f"Bước: {step}")
            solver.print_state(state)
    else:
        print("Không có lời giải!")

Đã tìm thấy đường đi sau 2 bước:
Bước: Start
 1  2  3  4
 5  6  7  8
 9 10  0 11
13 14 15 12
----------
Bước: Right
 1  2  3  4
 5  6  7  8
 9 10 11  0
13 14 15 12
----------
Bước: Down
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14 15  0
----------
